Make sure the right schema is used

In [0]:
USE CATALOG sac;

USE SCHEMA customer_service;

# Create bronze tables

In [0]:
CREATE TABLE IF NOT EXISTS customer_bronze (
        customer_id STRING,
        signup_date STRING,
        plan_tier STRING,
        address STRING,
        contract_type STRING,
        autopay_enabled STRING,
        account_age_months STRING,
        monthly_bill STRING,
        speed_tier_mbps STRING,
        data_usage_gb_last_month STRING,
        ingestion_time TIMESTAMP
    );

In [0]:
SELECT * FROM customer_bronze LIMIT 10

In [0]:
CREATE TABLE IF NOT EXISTS churn_bronze (
        customer_id STRING,
        churned STRING,
        churn_date STRING,
        churn_reason STRING,
        ingestion_time TIMESTAMP
    );

In [0]:
CREATE TABLE IF NOT EXISTS log_bronze (
        log_id STRING,
        customer_id STRING,
        timestamp STRING,
        speed_measured_mbps STRING,
        packet_loss_percent STRING,
        latency_ms STRING,
        downtime_minutes STRING,
        connection_drops_count STRING,
        issue_detected STRING,
        ingestion_time TIMESTAMP
    );

In [0]:
CREATE TABLE IF NOT EXISTS ticket_bronze (
        ticket_id STRING,
        customer_id STRING,
        timestamp_created STRING,
        timestamp_closed STRING,
        subject STRING,
        description STRING,
        category STRING,
        priority STRING,
        channel STRING,
        status STRING,
        solved_in_hours STRING,
        log_id STRING,
        technical_issue_type STRING,
        ingestion_time TIMESTAMP
    );

In [0]:
CREATE TABLE IF NOT EXISTS agent_bronze (
        agent_id STRING,
        first_name STRING,
        last_name STRING,
        employment_date STRING,
        experience_level STRING,
        employment_months STRING,
        monthly_salary_eur STRING,
        ingestion_time TIMESTAMP
    )

In [0]:
DROP TABLE chat;
CREATE TABLE IF NOT EXISTS chat_bronze (
        session_id STRING,
        customer_id STRING,
        agent_id STRING,
        timestamp_start STRING,
        timestamp_end STRING,
        messages STRING,
        resolution_status STRING,
        log_id STRING,
        chat_reason STRING,
        ingestion_time TIMESTAMP
    )

# Fill tables

In [0]:
COPY INTO customer_bronze FROM (
    SELECT
        *,
        current_timestamp() AS ingestion_time FROM
    '/Volumes/workspace/cs_stream/cs_stream/customer_profiles/'
) FILEFORMAT = CSV FORMAT_OPTIONS ('header' = 'true', 'multiLine' = 'true') COPY_OPTIONS ('mergeSchema' = 'true');

In [0]:
COPY INTO churn_bronze FROM (
    SELECT
        *,
        current_timestamp() AS ingestion_time FROM
    '/Volumes/workspace/cs_stream/cs_stream/churn_labels'
) FILEFORMAT = CSV FORMAT_OPTIONS ('header' = 'true') COPY_OPTIONS ('mergeSchema' = 'true')

In [0]:
COPY INTO log_bronze FROM (
    SELECT
        *,
        current_timestamp() AS ingestion_time FROM
    '/Volumes/workspace/cs_stream/cs_stream/connection_quality_logs'
) FILEFORMAT = CSV FORMAT_OPTIONS ('header' = 'true') COPY_OPTIONS ('mergeSchema' = 'true')

In [0]:
COPY INTO ticket_bronze FROM (
    SELECT
        *,
        current_timestamp() AS ingestion_time FROM
    '/Volumes/workspace/cs_stream/cs_stream/support_tickets'
) FILEFORMAT = CSV FORMAT_OPTIONS ('header' = 'true') COPY_OPTIONS ('mergeSchema' = 'true')

In [0]:
COPY INTO agent_bronze FROM (
    SELECT
        *,
        current_timestamp() AS ingestion_time FROM
    '/Volumes/workspace/cs_stream/cs_stream/agent_profiles'
) FILEFORMAT = CSV FORMAT_OPTIONS ('header' = 'true') COPY_OPTIONS ('mergeSchema' = 'true')

In [0]:
%python

%pip install pandas openpyxl

import pandas
import openpyxl

df = pandas.read_excel("/Volumes/workspace/cs_stream/cs_stream/chat_transcripts/chat_transcript.xlsx", engine='openpyxl')

print('Shape: ', df.shape)
print('Columns: ', df.columns)
print('Data Types: ', df.dtypes)

In [0]:
%python

from pyspark.sql.functions import current_timestamp

df = df.astype(str)
spark_df = spark.createDataFrame(df)

spark_df = spark_df.withColumn("ingestion_time", current_timestamp())

spark_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("chat_bronze")